# 01 — EDA and Feature Analysis

**Project**: AI-Driven Delivery Time Prediction  
**Dataset**: LaDe Shanghai Delivery Dataset (2023)  

This notebook covers:
1. Delivery duration distribution
2. Duration vs distance
3. Duration by hour
4. Duration by day of week
5. Courier historical performance
6. Distance distribution
7. Correlation analysis
8. Missing value analysis
9. Outlier analysis

In [ ]:
import sys
from pathlib import Path
ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
FIGURES_DIR = ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load cleaned + featurised dataset
df = pd.read_csv(ROOT / 'data' / 'processed' / 'cleaned_delivery_sh.csv',
                 parse_dates=['accept_time_dt'])
print(f'Loaded {len(df):,} records')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

---
## 1. Delivery Duration Distribution

The primary target variable. We expect a right-skewed distribution
typical of last-mile delivery operations.

In [ ]:
dur = df['delivery_duration_minutes']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Full distribution
axes[0].hist(dur, bins=120, color='#4A90D9', alpha=0.85, edgecolor='none')
axes[0].set_xlabel('Duration (minutes)')
axes[0].set_ylabel('Count')
axes[0].set_title('Full Range')
for p, c in [(0.5, 'red'), (0.75, 'orange'), (0.95, 'green')]:
    v = dur.quantile(p)
    axes[0].axvline(v, color=c, linestyle='--', linewidth=1.2, label=f'P{int(p*100)}={v:.0f}')
axes[0].legend(fontsize=9)

# Zoomed to <=360 min
axes[1].hist(dur[dur <= 360], bins=80, color='#5BAD6F', alpha=0.85, edgecolor='none')
axes[1].set_xlabel('Duration (minutes)')
axes[1].set_title('Zoomed (<=360 min)')

# Log scale to see tail
axes[2].hist(dur, bins=150, color='#E07B39', alpha=0.85, edgecolor='none', log=True)
axes[2].set_xlabel('Duration (minutes)')
axes[2].set_ylabel('Count (log)')
axes[2].set_title('Log Scale')

fig.suptitle('Delivery Duration Distribution', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_01_duration_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(dur.describe(percentiles=[.25,.5,.75,.90,.95,.99]))

**Observations**:
- Distribution is strongly right-skewed
- Median ~72 min, mean ~111 min — long right tail inflates the mean
- Most deliveries (75th pct) complete within ~120 min
- The tail beyond 360 min (6h) warrants investigation

---
## 2. Delivery Duration vs Straight-Line Distance

In [ ]:
# Sample 10k points to avoid overplotting
sample = df.sample(10000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(sample['straight_line_distance_km'], sample['delivery_duration_minutes'],
                alpha=0.2, s=8, color='#4A90D9', edgecolors='none')
axes[0].set_xlabel('Straight-Line Distance (km)')
axes[0].set_ylabel('Duration (minutes)')
axes[0].set_title('Duration vs Distance (10k sample)')
axes[0].set_ylim(0, 600)

# Bin by distance and show median duration
df['dist_bin'] = pd.cut(df['straight_line_distance_km'],
                         bins=[0, 1, 2, 3, 4, 5, 7, 10, 15, 100],
                         labels=['0-1','1-2','2-3','3-4','4-5','5-7','7-10','10-15','>15'])
bin_med = df.groupby('dist_bin', observed=True)['delivery_duration_minutes'].median()

axes[1].bar(range(len(bin_med)), bin_med.values, color='#5BAD6F', alpha=0.85)
axes[1].set_xticks(range(len(bin_med)))
axes[1].set_xticklabels(bin_med.index, rotation=30, ha='right')
axes[1].set_xlabel('Distance Bin (km)')
axes[1].set_ylabel('Median Duration (minutes)')
axes[1].set_title('Median Duration by Distance Bin')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_02_duration_vs_distance.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Correlation (distance vs duration): {df['straight_line_distance_km'].corr(df['delivery_duration_minutes']):.4f}")

**Observations**:
- Positive but moderate correlation between distance and duration
- High scatter at any given distance — courier speed and congestion vary a lot
- Median duration grows roughly linearly up to ~5km, then plateaus
- Haversine distance is a noisy proxy for actual road distance

---
## 3. Delivery Duration by Hour of Day

In [ ]:
hour_stats = df.groupby('hour')['delivery_duration_minutes'].agg(['median', 'mean', 'count'])

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(hour_stats.index, hour_stats['median'], 'o-', color='#4A90D9', linewidth=2, label='Median')
ax1.plot(hour_stats.index, hour_stats['mean'], 's--', color='#E07B39', linewidth=2, label='Mean')
ax2.bar(hour_stats.index, hour_stats['count'], alpha=0.2, color='gray', label='Order count')

ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Duration (minutes)', color='#333')
ax2.set_ylabel('Order Count', color='gray')
ax1.set_title('Delivery Duration by Hour of Day', fontsize=13)
ax1.set_xticks(range(24))
lines1, labels1 = ax1.get_legend_handles_labels()
ax1.legend(lines1, labels1, loc='upper left')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_03_duration_by_hour.png', dpi=120, bbox_inches='tight')
plt.show()

print(hour_stats.to_string())

**Observations**:
- Peak delivery hours: 9–11 AM and 2–5 PM
- Duration tends to be higher during rush-hour windows
- Very few deliveries at midnight–7 AM (courier rest)
- Evening deliveries (7–9 PM) show elevated durations

---
## 4. Delivery Duration by Day of Week

In [ ]:
dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_stats = df.groupby('day_of_week')['delivery_duration_minutes'].agg(['median','mean','count'])
dow_stats.index = dow_names[:len(dow_stats)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(dow_stats.index, dow_stats['median'], color='#4A90D9', alpha=0.85)
axes[0].set_xlabel('Day of Week')
axes[0].set_ylabel('Median Duration (minutes)')
axes[0].set_title('Median Delivery Duration by Day')

axes[1].bar(dow_stats.index, dow_stats['count'], color='#5BAD6F', alpha=0.85)
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Order Count')
axes[1].set_title('Order Volume by Day')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_04_duration_by_dow.png', dpi=120, bbox_inches='tight')
plt.show()

print(dow_stats)

**Observations**:
- Weekend deliveries show different volume patterns vs weekdays
- Median duration is relatively stable across days (~65-80 min)
- Day-of-week feature expected to have modest predictive power

---
## 5. Courier Historical Performance Distribution

In [ ]:
# Distribution of courier average delivery duration
courier_perf = df.groupby('courier_id').agg(
    total_deliveries=('delivery_duration_minutes','count'),
    avg_duration=('delivery_duration_minutes','mean'),
    median_duration=('delivery_duration_minutes','median'),
    on_time_rate=('delivery_duration_minutes', lambda x: (x <= 120).mean())
).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(courier_perf['total_deliveries'], bins=60, color='#4A90D9', alpha=0.85, edgecolor='none')
axes[0,0].set_xlabel('Total Deliveries per Courier')
axes[0,0].set_ylabel('# Couriers')
axes[0,0].set_title('Delivery Volume per Courier')

axes[0,1].hist(courier_perf['avg_duration'], bins=60, color='#E07B39', alpha=0.85, edgecolor='none')
axes[0,1].set_xlabel('Avg Delivery Duration (min)')
axes[0,1].set_ylabel('# Couriers')
axes[0,1].set_title('Courier Average Duration Distribution')

axes[1,0].hist(courier_perf['on_time_rate'], bins=40, color='#5BAD6F', alpha=0.85, edgecolor='none')
axes[1,0].set_xlabel('On-Time Rate (<= 120 min)')
axes[1,0].set_ylabel('# Couriers')
axes[1,0].set_title('Courier On-Time Rate Distribution')

axes[1,1].scatter(courier_perf['total_deliveries'], courier_perf['avg_duration'],
                  alpha=0.4, s=12, color='#7B68EE', edgecolors='none')
axes[1,1].set_xlabel('Total Deliveries')
axes[1,1].set_ylabel('Avg Duration (min)')
axes[1,1].set_title('Experience vs Speed')

fig.suptitle('Courier Performance Analysis', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_05_courier_performance.png', dpi=120, bbox_inches='tight')
plt.show()

print(courier_perf.describe())

**Observations**:
- Large variance in courier performance — some couriers average 2x the duration of others
- More experienced couriers tend to be faster (negative correlation visible)
- On-time rates cluster around 0.6–0.9 for most couriers

---
## 6. Straight-Line Distance Distribution

In [ ]:
dist = df['straight_line_distance_km']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(dist, bins=100, color='#9B59B6', alpha=0.85, edgecolor='none')
axes[0].set_xlabel('Distance (km)')
axes[0].set_ylabel('Count')
axes[0].set_title('Straight-Line Distance Distribution')
for p in [0.5, 0.75, 0.95]:
    v = dist.quantile(p)
    axes[0].axvline(v, color='red', linestyle='--', linewidth=1, alpha=0.7, label=f'P{int(p*100)}={v:.2f}km')
axes[0].legend(fontsize=9)

axes[1].hist(dist[dist < 20], bins=80, color='#1ABC9C', alpha=0.85, edgecolor='none')
axes[1].set_xlabel('Distance (km, <20)')
axes[1].set_title('Zoomed (<20 km)')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_06_distance_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(dist.describe(percentiles=[.25,.5,.75,.90,.95,.99]))

**Observations**:
- Most deliveries are within 5 km straight-line from dispatch location
- Long-distance outliers (>15 km) likely represent unusual assignments
- Distance distribution is right-skewed, consistent with typical urban courier patterns

---
## 7. Correlation Analysis

In [ ]:
numeric_cols = [
    'delivery_duration_minutes', 'hour', 'day_of_week', 'day_of_month',
    'month', 'is_weekend', 'straight_line_distance_km', 'aoi_type', 'region_id',
    'courier_previous_delivery_count', 'courier_avg_delivery_duration',
    'courier_median_delivery_duration', 'courier_std_delivery_duration',
    'courier_avg_recent_delivery_duration', 'courier_on_time_rate',
    'route_previous_count', 'route_avg_delivery_duration',
    'route_avg_recent_delivery_duration'
]
numeric_cols = [c for c in numeric_cols if c in df.columns]

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix', fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_07_correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

# Print top correlations with target
target_corr = corr['delivery_duration_minutes'].drop('delivery_duration_minutes').abs().sort_values(ascending=False)
print('Top correlations with delivery_duration_minutes:')
print(target_corr.head(12).to_string())

**Observations**:
- Courier historical duration features are most strongly correlated with target
- distance has a moderate positive correlation
- Hour and day_of_week show weaker linear correlations (non-linear effects likely)
- High correlation between courier_avg and courier_median expected — multicollinearity
  is handled by tree-based models naturally

---
## 8. Missing Value Analysis

In [ ]:
# Check missing values in features after historical computation
# The first delivery of each courier will have NaN historical features
hist_cols = [
    'courier_previous_delivery_count', 'courier_avg_delivery_duration',
    'courier_median_delivery_duration', 'courier_std_delivery_duration',
    'courier_avg_recent_delivery_duration', 'courier_on_time_rate',
    'route_previous_count', 'route_avg_delivery_duration',
    'route_avg_recent_delivery_duration'
]
hist_cols = [c for c in hist_cols if c in df.columns]

miss_counts = df[hist_cols].isnull().sum()
miss_pct = (df[hist_cols].isnull().mean() * 100).round(2)

miss_df = pd.DataFrame({'Missing Count': miss_counts, 'Missing %': miss_pct})
print('Historical Feature Missing Values:')
print(miss_df.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
miss_pct_sorted = miss_pct.sort_values(ascending=True)
ax.barh(miss_pct_sorted.index, miss_pct_sorted.values, color='#E07B39', alpha=0.85)
ax.set_xlabel('Missing %')
ax.set_title('Missing Values in Historical Features (first delivery = NaN)')
for i, (val, idx) in enumerate(zip(miss_pct_sorted.values, miss_pct_sorted.index)):
    ax.text(val + 0.1, i, f'{val:.1f}%', va='center', fontsize=9)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_08_missing_values.png', dpi=120, bbox_inches='tight')
plt.show()

**Observations**:
- Missing values in historical features arise from the cold-start problem
- courier_std has more NaN than other courier features (requires >=2 prior deliveries for std)
- Route features have more NaN than courier features — many AOIs have few historical visits
- These NaN are filled with training-set medians during model training (fit_nan_fill_values)

---
## 9. Outlier Analysis

In [ ]:
dur = df['delivery_duration_minutes']

# Box plots by hour
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot (duration by hour)
hour_groups = [df[df['hour']==h]['delivery_duration_minutes'].clip(upper=500).values
               for h in sorted(df['hour'].unique())]
axes[0].boxplot(hour_groups, labels=sorted(df['hour'].unique()),
                flierprops={'markersize': 2, 'alpha': 0.3})
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Duration (min, clipped at 500)')
axes[0].set_title('Duration Spread by Hour (Outliers Visible)')

# Percentile analysis
pcts = np.arange(90, 100.1, 0.5)
pct_vals = [dur.quantile(p/100) for p in pcts]
axes[1].plot(pcts, pct_vals, 'o-', color='#4A90D9', linewidth=2)
axes[1].axhline(1440, color='red', linestyle='--', linewidth=1.5, label='24h cap (1440 min)')
axes[1].set_xlabel('Percentile')
axes[1].set_ylabel('Duration (minutes)')
axes[1].set_title('Upper Tail Percentile Analysis')
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_09_outlier_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

print('Percentile analysis of duration:')
for p in [90, 95, 97, 98, 99, 99.5, 99.9, 100]:
    v = dur.quantile(p/100)
    print(f'  P{p:5.1f}: {v:.1f} min')

print(f'\n  Records removed by 24h cap: {(dur > 1440).sum():,}'
      f' ({(dur > 1440).mean()*100:.3f}% of cleaned data)')

**Observations**:
- After cleaning (zero durations, bad GPS removed), 24h cap removes <0.25% of data
- The tail is extreme — the max was 47,739 min (~33 days) indicating open/unresolved orders
- Outlier cap of 1440 min is conservative and well-justified for last-mile delivery
- Hour-level box plots show consistent spreads with no single problematic hour

---
## Summary

| Observation | Implication |
|---|---|
| Right-skewed duration | XGBoost handles this natively; MAE preferred over RMSE |
| Hour/day effects non-linear | Tree models outperform linear regression |
| Courier history highly correlated | Critical feature group — leakage prevention essential |
| Distance moderate correlation | Useful but not dominant; GPS noise limits its power |
| High courier-to-courier variance | Per-courier historical features should improve accuracy |
| Route (AOI) variance | Route history adds signal beyond time+distance features |